# 🕵️ Dia 46 — Escrevendo um informe de inteligência com linguagem calibrada

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 46* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-46.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

7h40, segunda-feira. Marta Quintão tem conselho às 9h e quer **uma página**. Rafa leu a sua primeira frase, *"O Pescador possivelmente pode estar expandindo a campanha"*, e mandou reescrever: *"Possivelmente quanto? E quem é 'o Pescador'? Até ontem era um cluster."*

Neste notebook você vai:

1. **Ver as duas escalas como dados** (ICD 203 e PHIA), inclusive as lacunas da PHIA.
2. **Jogar o quiz "quanto é provável?"**.
3. **Construir um verificador de calibração** que implementa o fluxograma do passo 4.
4. **Escrever o informe para a Marta** por formulário e gerar `brief.md` (e `.docx` se houver Pandoc).

🐍 Só biblioteca padrão (o Pandoc é opcional).

## 1. As duas escalas, como dados

Cada termo vira uma faixa numérica. Com isso o código consegue dizer qual termo corresponde a uma porcentagem e **onde a PHIA deixa buracos de propósito**.

In [ ]:
# (termo em pt-BR, termo original, mínimo %, máximo %)
ICD203 = [
    ("quase nenhuma chance", "almost no chance / remote", 1, 5),
    ("muito improvável", "very unlikely / highly improbable", 5, 20),
    ("improvável", "unlikely / improbable", 20, 45),
    ("chance mais ou menos igual", "roughly even chance / odds", 45, 55),
    ("provável", "likely / probable", 55, 80),
    ("muito provável", "very likely / highly probable", 80, 95),
    ("quase certo", "almost certain / nearly certain", 95, 99),
]
PHIA = [
    ("chance remota", "remote chance", 0, 5),
    ("altamente improvável", "highly unlikely", 10, 20),
    ("improvável", "unlikely", 25, 35),
    ("possibilidade realista", "realistic possibility", 40, 50),
    ("provável", "likely / probable", 55, 75),
    ("altamente provável", "highly likely", 80, 90),
    ("quase certo", "almost certain", 95, 100),
]

def desenhar_escala(nome, escala):
    """Uma barra de 0 a 100%: cada caractere = 2 pontos percentuais."""
    barra = [" "] * 50
    for i, (_, _, lo, hi) in enumerate(escala):
        for p in range(lo // 2, min(50, (hi + 1) // 2)):
            barra[p] = "▁▂▃▄▅▆▇"[i]
    print(f"{nome:<8}|{''.join(barra)}|  (espaço em branco = lacuna)")

print("         0%" + " " * 20 + "50%" + " " * 21 + "100%")
desenhar_escala("ICD 203", ICD203)
desenhar_escala("PHIA", PHIA)

def termo_para(pct, escala):
    """Que termo usar para uma probabilidade? Na lacuna, devolve None: comprometa-se com um lado."""
    for pt, en, lo, hi in escala:
        if lo <= pct <= hi:
            return f"{pt} ({en})"
    return None

for pct in (3, 22, 38, 50, 77, 92):
    print(f"{pct:>3}% → ICD 203: {termo_para(pct, ICD203)!s:<45} PHIA: {termo_para(pct, PHIA)}")

> 💡 Repare no `38%` e no `92%`: na PHIA eles caem numa **lacuna**. A régua não deixa você ficar em cima do muro: ou é "improvável" (25-35%), ou "possibilidade realista" (40-50%). Decida e justifique.

## 2. 🎮 Quiz: quanto é provável?

Para cada situação do caso, escolha o termo **PHIA** que você usaria. Digite o número: **1** chance remota · **2** altamente improvável · **3** improvável · **4** possibilidade realista · **5** provável · **6** altamente provável · **7** quase certo.

In [ ]:
SITUACOES = [
    ("serraazul-pix e serraazul-seguranca estão no MESMO certificado TLS (SAN). Mesmo operador?", "7",
     "Um único certificado lista os dois nomes: quem emitiu controla os dois. ~95%+."),
    ("coopserra-azul.example só se parece no nome, está estacionado e sem kit. É do cluster?", "3",
     "Semelhança lexical é pivô fraco; domínios parecidos são registrados por oportunistas também. ~25-35%."),
    ("As ondas 4 e 5 (encurtador, servidor e certificado diferentes) são de outro operador que não o das ondas 1-3?", "5",
     "Três diferenças independentes apontam para o mesmo lado, mas o kit é idêntico. ~55-75%."),
    ("A Serra Azul será alvo de uma nova onda do IscaKit nos próximos 30 dias?", "4",
     "Já foi alvo e o kit segue ativo, mas houve derrubada e alerta público. Perto de 40-50%: diga as razões."),
    ("O IscaKit tem relação com o ransomware do print que assustou o conselho?", "1",
     "Nenhuma evidência liga os dois; são modalidades diferentes. Nem 'altamente improvável': não há base alguma."),
]
NOMES = {str(i + 1): t[0] for i, t in enumerate(PHIA)}
MODO_JOGO = True

acertos = 0
for i, (sit, certo, porque) in enumerate(SITUACOES, start=1):
    print(f"\n[{i}/{len(SITUACOES)}] {sit}")
    if MODO_JOGO:
        r = ""
        while r not in NOMES:
            r = input("   Termo PHIA (1-7): ").strip()
        perto = abs(int(r) - int(certo)) <= 1                    # vizinho também vale como discussão
        acertos += (r == certo)
        print("   ✅" if r == certo else ("   🟨 vizinho: defenda por escrito" if perto else "   ❌"))
    print(f"   Sugerido: {NOMES[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(SITUACOES)} exatos. Calibração é discussão: o importante é a justificativa.")

## 3. O verificador de calibração

Implementa o fluxograma do passo 4 do original, frase por frase. Ele **sinaliza**; quem decide é você. Regras:

| Regra | De onde vem |
|---|---|
| Palavra evasiva em julgamento ("pode", "poderia", "talvez", "possivelmente" / *may, might, could, possibly*) | passo 4 |
| Termos de duas escalas no mesmo texto | Conceito: "escolha uma escala" |
| Dois termos de probabilidade na mesma frase ("provavelmente possível") | passo 4 |
| Probabilidade e confiança na mesma frase | ICD 203, D.6.e.(2)(b) |
| Julgamento ("avaliamos") sem nível de confiança numa frase seguinte do mesmo item | Checkpoint |
| "Definitivamente", "com certeza", "sem dúvida" (certeza sem escala) | passo 2, primeira linha da tabela |
| Nome "Pescador" como sujeito | ➕ regra do caso |

In [ ]:
import re

EVASIVAS = re.compile(r"\b(pode|podem|poderia|poderiam|talvez|possivelmente|eventualmente|"
                      r"may|might|could|possibly)\b", re.I)
CONFIANCA = re.compile(r"\bconfian[çc]a\b|\bconfidence\b", re.I)
CERTEZA_SEM_ESCALA = re.compile(r"(?<!quase )\b(definitivamente|com certeza|sem d[uú]vida|certamente|definitely|certainly)\b", re.I)
JULGAMENTO = re.compile(r"\bavaliamos\b|\bwe assess\b", re.I)

def variantes(termo):
    """'quase certo' também aparece como 'quase certamente'; 'provável' como 'provavelmente'."""
    v = {termo}
    if termo.endswith("vel"):
        v.add(termo[:-3] + "velmente")                  # improvável → improvavelmente
        v.add(termo.replace("á", "a")[:-3] + "velmente")
    if termo.endswith("o"):
        v.add(termo[:-1] + "amente")                    # quase certo → quase certamente
    return v

def termos_na_frase(frase, escala):
    """Acha termos da escala, do mais longo para o mais curto ('altamente provável' antes de 'provável')."""
    achados, texto = [], frase.lower()
    for pt, _, _, _ in sorted(escala, key=lambda t: -len(t[0])):
        padrao = r"\b(" + "|".join(re.escape(v) for v in variantes(pt)) + r")\b"
        if re.search(padrao, texto):
            achados.append(pt)
            texto = re.sub(padrao, " ", texto)          # evita contar 'provável' dentro de 'altamente provável'
    return achados

SO_ICD = {t[0] for t in ICD203} - {t[0] for t in PHIA}   # termos exclusivos de cada escala
SO_PHIA = {t[0] for t in PHIA} - {t[0] for t in ICD203}

def verificar(texto, escala_declarada="PHIA"):
    # Quebramos em blocos (linhas/itens) e depois em frases, guardando o resto do bloco:
    # a confiança pode vir em qualquer frase SEGUINTE do mesmo item, nunca na mesma frase.
    frases, resto_do_bloco = [], []
    for bloco in texto.split("\n"):
        partes = [f.strip() for f in re.split(r"(?<=[.!?])\s+", bloco) if f.strip()]
        for j, f in enumerate(partes):
            frases.append(f)
            resto_do_bloco.append(" ".join(partes[j + 1:]))
    alertas, escalas_vistas = [], set()
    for i, f in enumerate(frases):
        if f.startswith(("#", "---", "title:", "subtitle:", "!", "- Revisar", "- Bloquear")):
            continue                                     # cabeçalhos e ações não são julgamentos
        termos = termos_na_frase(f, ICD203 + PHIA)
        escalas_vistas |= {"ICD 203" for t in termos if t in SO_ICD} | {"PHIA" for t in termos if t in SO_PHIA}
        e = EVASIVAS.search(f)
        if e:                                            # sinaliza sempre; você decide se é julgamento
            alertas.append((f, f"palavra evasiva '{e.group(0)}': se for julgamento, troque por um termo da escala"))
        c = CERTEZA_SEM_ESCALA.search(f)
        if c:
            alertas.append((f, f"'{c.group(0)}' é certeza sem escala: use um termo da escala ou atribua à fonte"))
        if len(set(termos)) > 1:
            alertas.append((f, f"dois termos de probabilidade: {termos}"))
        if termos and CONFIANCA.search(f):
            alertas.append((f, "probabilidade e confiança na MESMA frase (ICD 203)"))
        if JULGAMENTO.search(f):
            if not termos:
                alertas.append((f, "julgamento sem termo de probabilidade"))
            if not CONFIANCA.search(resto_do_bloco[i]):
                alertas.append((f, "julgamento sem nível de confiança numa frase seguinte do mesmo item"))
        if re.match(r"^(\d+\.\s*)?(o |the )?pescador\b", f, re.I):
            alertas.append((f, "'Pescador' como sujeito: o sujeito é o cluster CA-ISCA-01"))
    if len(escalas_vistas) > 1:
        alertas.append(("(texto inteiro)", "mistura de escalas: " + " + ".join(sorted(escalas_vistas))))
    elif escalas_vistas and escala_declarada not in escalas_vistas:
        alertas.append(("(texto inteiro)", f"escala declarada {escala_declarada}, mas o texto usa {escalas_vistas}"))
    return alertas

def relatorio(texto, escala="PHIA"):
    a = verificar(texto, escala)
    if not a:
        print("✅ Nenhum alerta de calibração.")
    for frase, motivo in a:
        print(f"⚠️  {motivo}\n     → {frase[:110]}")
    return a

print("Rascunho de sexta-feira:\n")
_ = relatorio("O Pescador possivelmente pode estar expandindo a campanha para outros estados. "
              "Avaliamos que é altamente provável que os domínios sejam do mesmo ator, com confiança alta. "
              "Também é muito provável que o IP esteja relacionado.")

### As quatro frases do original

Rode o verificador nas versões "antes" e "depois" da tabela do passo 2 (em pt-BR). O "depois" ainda pode gerar alertas: leia cada um e decida se é falso positivo.

In [ ]:
ANTES_DEPOIS = [
    ("Os domínios são definitivamente parte da mesma campanha.",
     "Avaliamos que os três domínios são quase certamente operados pelo mesmo ator: um único certificado lista os três nomes (crt.sh, consultado em 2026-09-30). Confiança: alta."),
    ("198.51.100.23 pode estar relacionado.",
     "Avaliamos que é uma possibilidade realista que 198.51.100.23 pertença ao mesmo cluster. O único vínculo é um favicon compartilhado encontrado em 12 hosts, e não temos um segundo pivô. Confiança: baixa."),
    ("Este ator é sofisticado e perigoso.", "(Cortar.)"),
    ("O ator possivelmente poderia nos atacar em breve.",
     "Não temos informação que indique que a nossa organização é alvo. O relatório semente descreve a mira em <setor> em <região> durante <meses>."),
]
for antes, depois in ANTES_DEPOIS:
    print("ANTES :", antes); relatorio(antes)
    print("DEPOIS:", depois[:90] + "..."); relatorio(depois)
    print("-" * 60)

> 📝 O segundo "depois" mostra um ponto fino: o original põe "Confiança: baixa." **na frase seguinte**, exatamente para cumprir a regra do ICD 203. O verificador procura a confiança nas frases seguintes do mesmo item por isso. E as frases "antes" agora acendem alertas: é o objetivo.

## 4. ➕ O informe para a Marta

Preencha o formulário. Os valores padrão são um rascunho **razoável, não perfeito**: rode, leia os alertas e melhore. A célula monta o `brief.md` no modelo pt-BR do README, conta as palavras (meta: 400 a 550) e roda o verificador.

In [ ]:
#@title Informe CA-ISCA-01 { display-mode: "form" }
essencial = "O golpe do 'Pix bloqueado' contra a Serra Azul faz parte de um cluster de phishing (CA-ISCA-01) que atingiu outras quatro cooperativas com o mesmo kit; avaliamos como provável que a Serra Azul volte a ser alvo enquanto o kit estiver ativo. Temos confiança moderada nisso (base forte sobre o kit, fraca sobre quem o opera), e a ação mais importante hoje é reenviar aos cooperados o alerta oficial sobre SMS de 'Pix bloqueado', antes da próxima janela de disparos (07h às 09h)."  #@param {type:"string"}
j1 = "Avaliamos que é quase certo que serraazul-pix[.]example e serraazul-seguranca[.]example são controlados pelo mesmo operador, porque um único certificado TLS lista os dois nomes."  #@param {type:"string"}
j1_conf = "Confiança: alta (certificado observado no crt.sh em 2026-10-07)."  #@param {type:"string"}
j2 = "Avaliamos como provável que as ondas 4 e 5 tenham operação distinta das ondas 1 a 3, porque encurtador, servidor e certificado mudaram, embora o kit seja o mesmo."  #@param {type:"string"}
j2_conf = "Confiança: moderada (três pivôs, nenhum confirmado por segunda fonte independente)."  #@param {type:"string"}
j3 = "Não sabemos se as ondas 4 e 5 são de compradores independentes do kit ou de imitadores; a análise de hipóteses concorrentes não separa essas duas explicações."  #@param {type:"string"}
j3_conf = "Confiança: baixa em qualquer resposta a essa pergunta."  #@param {type:"string"}
evidencia = "Semente: SMS recebidos pelos cooperados em 2026-10-05 (07h12–08h40 BRT). Pivôs: SAN de certificado compartilhado (forte), hash de corpo HTML do kit com 3 resultados (médio), favicon customizado com 9 resultados (médio). Ondas 4 e 5: encurtador diferente, servidor 198.51.100[.]40 e certificado sem SAN compartilhado com as ondas 1 a 3. Reclamações: 37 nas duas primeiras horas na Serra Azul; 3 cooperados com transferências Pix não reconhecidas, tratadas pela própria cooperativa. O apelido 'pescador' aparece como comentário no código do kit, presente nas cinco ondas, e não identifica ninguém. Figura 1: grafo do cluster (Dia 40); matriz de hipóteses concorrentes (Dia 45) em anexo."  #@param {type:"string"}
nao_mostra = "Esta análise não mostra quem opera o cluster, se haverá nova onda contra a Serra Azul em data específica, nem qualquer relação com ransomware, que não foi demonstrada."  #@param {type:"string"}
acoes = "Bloquear e alertar nos indicadores do bundle cluster-iscakit.json (TLP:AMBER) até 2026-12-31; reenviar o alerta aos cooperados até 2026-10-13; pedir ao provedor de SMS da cooperativa o bloqueio de remetentes que imitem a marca; orientar o atendimento a não confirmar dados de cooperados por telefone ou mensagem; revisar esta avaliação em 2026-10-26."  #@param {type:"string"}
mudaria = "Um novo certificado com os nomes das ondas 4 e 5 junto com os das ondas 1 a 3 (aumentaria a confiança de operador único); o mesmo hash do kit em sites sem relação com cooperativas (indicaria kit vendido amplamente)."  #@param {type:"string"}
tlp = "TLP:AMBER"  #@param ["TLP:RED", "TLP:AMBER+STRICT", "TLP:AMBER", "TLP:GREEN", "TLP:CLEAR"]
data = "2026-10-12"  #@param {type:"string"}

BRIEF = f"""---
title: "Cluster CA-ISCA-01 (golpe do 'Pix bloqueado'): avaliação e ações"
subtitle: "{tlp} | {data} | Termos de probabilidade: régua PHIA"
---

## Essencial
{essencial}

## Julgamentos-chave
1. {j1} {j1_conf}
2. {j2} {j2_conf}
3. {j3} {j3_conf}

## Evidência
{evidencia}

## O que isto não mostra
{nao_mostra}

## Ações recomendadas
{acoes}

## Indicadores que mudariam esta avaliação
{mudaria}
"""

corpo = BRIEF.split("---", 2)[2]                       # conta palavras sem o cabeçalho YAML
n_palavras = len(re.findall(r"\w+", corpo))
print(f"Palavras: {n_palavras} " + ("✅ (400-550)" if 400 <= n_palavras <= 550 else "⚠️ meta: 400 a 550"))

# Checagens do checkpoint que o verificador não cobre:
duas_primeiras = " ".join(re.split(r"(?<=[.!?])\s+", essencial)[:2]).lower()
print("BLUF diz o quão seguro:", "✅" if termos_na_frase(duas_primeiras, PHIA + ICD203) else "❌")
print("BLUF diz a ação:", "✅" if re.search(r"a(ç|c)[aã]o", duas_primeiras) else "❌ (nomeie a ação)")
print("Algum julgamento sobre o que não sabemos:", "✅" if "não sabemos" in (j1 + j2 + j3).lower() else "❌")
print("Indicadores neutralizados ([.]):", "✅" if ".example" not in BRIEF.replace("[.]example", "") else "❌ há domínio sem [.]")
print()
ALERTAS = relatorio(BRIEF, "PHIA")

from pathlib import Path
Path("brief.md").write_text(BRIEF, encoding="utf-8")
print("\n💾 brief.md gravado")

### Renderizar com Pandoc (opcional)

No Colab, a primeira linha instala o Pandoc. Fora dele, a célula verifica se o Pandoc existe e, se não, só avisa.

In [ ]:
!apt-get -qq install -y pandoc > /dev/null
import shutil
import subprocess

if shutil.which("pandoc"):
    subprocess.run(["pandoc", "brief.md", "-o", "brief.docx"], check=False)
    print("💾 brief.docx gerado. Abra e confira se cabe numa página com a figura.")
else:
    print("Pandoc não encontrado: brief.md está pronto; rode 'pandoc brief.md -o brief.docx' onde houver Pandoc.")

try:
    from google.colab import files
    files.download("brief.md")
    if shutil.which("pandoc"):
        files.download("brief.docx")
except ImportError:
    pass

## ✅ Antes de fechar o Dia 46

- [ ] Você sabe explicar a diferença entre probabilidade e confiança sem consultar notas
- [ ] As quatro frases do original reescritas (e verificadas)
- [ ] `brief.md` para a Marta com 400–550 palavras e sem alertas que você não consiga justificar
- [ ] Nenhuma afirmação sobre **quem** está por trás do cluster
- [ ] (Original) `brief.md`/`brief.docx` do cluster do Dia 40 e a cópia marcada para calibração

**Amanhã (Dia 47):** o conselho perguntou "mas quem é o Pescador?". Você vai aprender por que ferramenta compartilhada não é assinatura, graduando cada evidência pelo custo de falsificá-la.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>